In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
from aquacrop import AquaCropModel, Soil, Crop, InitialWaterContent
from aquacrop.utils import prepare_weather, get_filepath


In [ ]:
import rasterio as rio
import geopandas as gpd
import numpy as np

In [ ]:
# repository root (notebooks live in <repo>/notebooks)
base_loc = os.path.abspath(os.path.join(os.getcwd(), '..'))
weather_loc=os.path.join(base_loc,'data','weather_aquacrop')
soil_loc=os.path.join(base_loc,'data','soil')

In [ ]:
def get_weather(dist):
    filepath=get_filepath(os.path.join(weather_loc,dist+'.txt'))
    weather_data = prepare_weather(filepath)
    return weather_data

In [ ]:
# Rajasthan district boundaries (subset of the all-India district shapefile)
india_shape = os.path.join(base_loc, 'data', 'location data', 'Rajasthan.shp')
data=gpd.read_file(india_shape)
raj=data[data['STATE_NAME']=='Rajasthan']
raj=raj.drop(columns=['Crop','ID'], errors='ignore')

In [ ]:
#raj.to_file(os.path.join(base_loc,'data','location data','Rajasthan.shp'))

In [ ]:
def getFeatures(gdf):
    import json
    return [json.loads(gdf.to_json())['features'][0]['geometry']]

In [ ]:
def clip_raster(rasObj,vecObj):
    from rasterio.mask import mask
    cord=getFeatures(vecObj)
    output,trans=mask(rasObj,cord,crop=True)
    meta=rasObj.meta
    meta.update(transform=trans,height=int(output.shape[1]), width=int(output.shape[2]),nodata=0)
    return output


In [ ]:
def image_process(sand,clay,oc,shape):
    img_sand=clip_raster(rio.open(sand),shape)
    img_clay=clip_raster(rio.open(clay),shape)
    img_oc=clip_raster(rio.open(oc),shape)
    s=round(np.mean(img_sand*(1000**-1))*100,2)
    c=round(np.mean(img_clay*(1000**-1))*100,2)
    o=round(np.mean(img_oc*(10000**-1))*100,2)
    return (s,c,o)

In [ ]:
def get_soil(dist):
    area=raj[raj['DISTRICT']==dist]
    sand=os.path.join(soil_loc,'Rajasthan_sand_0-5cm_mean.tif')
    clay=os.path.join(soil_loc,'Rajasthan_clay_0-5cm_mean.tif')
    oc=os.path.join(soil_loc,'Rajasthan_soc_0-5cm_mean.tif')
    soil_val=image_process(sand,clay,oc,area)
    custom = Soil('custom',cn=46,rew=7)
    custom.add_layer_from_texture(thickness=custom.zSoil,
                                  Sand=soil_val[0],Clay=soil_val[1],
                                  OrgMat=soil_val[2],penetrability=100)
    return custom

In [ ]:
InitWC = InitialWaterContent(value=['FC'])

In [ ]:
def model_data(pm,dist):
    model = AquaCropModel(sim_start_time=f'{1990}/06/01',
                        sim_end_time=f'{2020}/10/15',
                        weather_df=get_weather(dist),
                        soil=get_soil(dist),
                        crop=pm,
                        initial_water_content=InitWC)
    return model

In [ ]:
def crop_data(planting_date,d):
    pm = Crop('custom',
      planting_date=planting_date,
      harvest_date='10/15',
      CropType=3,
      PlantMethod=1,
      CalendarType=1,
      SwitchGDD=0,
      EmergenceCD=4,
      MaxRootingCD=45,
      SenescenceCD=75,
      MaturityCD=105,
      HIstartCD=35,
      FloweringCD=39,
      YldFormCD=66,
      GDDmethod=3,
      Tbase=8,
      Tupp=32,
      PolHeatStress=1,
      Tmax_up=40,
      Tmax_lo=45,
      PolColdStress=1,
      Tmin_up=8,
      Tmin_lo=3,
      TrColdStress=1,
      GDD_up=11,
      GDD_lo=0,
      Zmin=0.3,
      Zmax=1.75,
      fshape_r=1.8,
      SxTopQ=0.051,
      SxBotQ=0.013,
      SeedSize=5,
      PlantPop=55556,
      CCx=0.95,
      CDC_CD=0.096,
      CGC_CD=0.269,
      Kcb=1.1,
      fage=0.3,
      WP=14,
      WPy=100,
      fsink=0.5,
      HI0=0.27,
      dHI_pre=0,
      a_HI=0.5,
      b_HI=10,
      dHI0=40,
      Determinant=1,
      exc=50,
      p_up1=0.32,
      p_up2=0.75,
      p_up3=0.58,
      p_up4=0.92,
      p_lo1=0.66,
      p_lo2=1,
      p_lo3=1,
      p_lo4=1,
      fshape_w1=3,
      fshape_w2=3,
      fshape_w3=3,
      fshape_w4=4.5,
      fshape_b=13.8135,
      PctZmin=70,
      fshape_ex=-6,
      ETadj=1,
      Aer=6,
      LagAer=3,
      beta=12,
      a_Tr=1,
      GermThr=0.2,
      CCmin=0.05,
      MaxFlowPct=33.3,
      HIini=0.01,
      bsted=0.000138,
      bface=0.001165)
    return model_data(pm,d)

In [ ]:
def output(model,planting_date,d):
    planting_date=planting_date.replace('/','-')
    (model._outputs.final_stats).to_excel(os.path.join(base_loc,'output','Final_stats_'+planting_date+'-'+d+'.xlsx'))
    #model._outputs.water_flux.to_excel(os.path.join(base_loc,'output','water_flux_'+planting_date+'-'+d+'.xlsx'))
    #model._outputs.water_storage.to_excel(os.path.join(base_loc,'output','water_storage_'+planting_date+'-'+d+'.xlsx'))
    #model._outputs.crop_growth.to_excel(os.path.join(base_loc,'output','crop_growth_'+planting_date+'-'+d+'.xlsx'))

In [ ]:
districts=list(raj['DISTRICT'].unique())

In [ ]:
for dist in districts:
    for d in np.arange(1,8):
        d=int((d-1)*15)
        init_date=datetime.strptime('01-06','%d-%m')
        date=init_date+timedelta(days=d)
        planting_date=(datetime.strftime(date,'%m/%d'))
        model=crop_data(planting_date,dist)
        model.run_model(till_termination=True)
        output(model,planting_date,dist)

In [ ]:
for dist in districts:
    req=pd.DataFrame()
    for d in np.arange(1,8):
        d=int((d-1)*15)
        init_date=datetime.strptime('01-06','%d-%m')
        date=init_date+timedelta(days=d)
        planting_date=(datetime.strftime(date,'%B %d'))
        plt_dt=datetime.strftime(date,'%m-%d')
        fn=os.path.join(base_loc,'output','Final_stats_'+plt_dt+'-'+dist+'.xlsx')
        df=pd.read_excel(fn)
        req[planting_date]=df['Yield (tonne/ha)']
    fig=req.plot.box().get_figure()
    fn=os.path.join(base_loc,'Graphs','Planting_date vs Yield ('+dist+').png')
    plt.title('Planting_date vs Yield ('+dist+')')
    plt.ylabel('Yield(t/Ha)')
    fig.savefig(fn,dpi=300,bbox_inches='tight')